In [ ]:
import cv2
import numpy as np
import os

In [8]:
from tqdm import tqdm

# Use cat-dog-bird dataset from Kaggle
# https://www.kaggle.com/datasets/mahmoudnoor/high-resolution-catdogbird-image-dataset-13000

dataset_dir = "cat-dog-bird"

# Load first 100 images from each class and generate feature_points
features = []
sift = cv2.SIFT_create()
num_ims = 100

# Build features for bag of words
for label in ['cat', 'dog', 'bird']:
    class_dir = os.path.join(dataset_dir, label, label)
    for filename in tqdm(os.listdir(class_dir)[:num_ims]):
        img_path = os.path.join(class_dir, filename)
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptor = sift.detectAndCompute(gray, None)
        if descriptor is not None:
            for d in descriptor:
                features.append(d)

100%|██████████| 100/100 [00:15<00:00,  6.56it/s]


In [9]:
features = np.array(features)
print(f"Total feature points: {features.shape[0]}")

Total feature points: 2186516


In [ ]:
from sklearn.cluster import MiniBatchKMeans

# Cluster feature points using KMeans to make visual words
num_clusters = 50
kmeans = MiniBatchKMeans(n_clusters=num_clusters)
kmeans.fit(features)
visual_words = kmeans.cluster_centers_

Visual words shape: (50, 128)


In [25]:
# Separate images into training and testing sets
train_data = []
test_data = []
train_labels = []
test_labels = []

num_ims = 500

for label in ['cat', 'dog', 'bird']:
    class_dir = os.path.join(dataset_dir, label, label)
    images = os.listdir(class_dir)[:num_ims]
    split_index = int(0.8 * len(images))
    train_images = images[:split_index]
    test_images = images[split_index:]

    for filename in tqdm(train_images):
        img_path = os.path.join(class_dir, filename)
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptor = sift.detectAndCompute(gray, None)
        if descriptor is not None:
            histogram = np.zeros(num_clusters)
            preds = kmeans.predict(descriptor)
            for p in preds:
                histogram[p] += 1
            train_data.append(histogram)
            train_labels.append(label)

    for filename in tqdm(test_images):
        img_path = os.path.join(class_dir, filename)
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptor = sift.detectAndCompute(gray, None)
        if descriptor is not None:
            histogram = np.zeros(num_clusters)
            preds = kmeans.predict(descriptor)
            for p in preds:
                histogram[p] += 1
            test_data.append(histogram)
            test_labels.append(label)

  0%|          | 0/400 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:17<00:00,  5.59it/s]


In [26]:
train_data = np.array(train_data)
test_data = np.array(test_data)
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print(train_data.shape, test_data.shape)

# Train SVM classifier
from sklearn import svm
clf = svm.SVC()
clf.fit(train_data, train_labels)

(1200, 50) (300, 50)


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [27]:
# Evaluate classifier on the test set
preds = clf.predict(test_data)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy = accuracy_score(test_labels, preds)
conf_matrix = confusion_matrix(test_labels, preds)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.54
Confusion Matrix:
[[81  6 13]
 [52 34 14]
 [38 15 47]]
